In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Update the imports to include the new Free Agent function
from espn_pull import fetch_espn_data, fetch_espn_free_agents
from fangraphs_pull import fetch_fangraphs_projections

# ==========================================
# 1. CONFIGURATION & DATA REFRESH
# ==========================================
MY_TEAM_NAME = "Budget Ballers"
LEAGUE_ID = 283187668  
YEAR = 2026

# Pathlib safely handles the home directory and slashes
FOLDER = Path.home() / 'Documents' / 'Python Projects' / 'Fantasy_Baseball_Model'
TOP_N_FA = 200

print(f"--- STARTING REST-OF-SEASON (ROS) ANALYSIS FOR {MY_TEAM_NAME} ---")

# Execute the data pulls to ensure we have the freshest data before analyzing
print("\n[Updating Data Pipelines...]")

# A. Pull FanGraphs Projections
df_hitters_proj, df_pitchers_proj = fetch_fangraphs_projections("steamer_ros")

# B. Pull ESPN Rosters and Standings
df_rosters, df_current = fetch_espn_data(LEAGUE_ID, YEAR)

# C. Pull the REAL ESPN Free Agent List (New!)
df_fa = fetch_espn_free_agents(LEAGUE_ID, YEAR)

print("[Data Pipelines Updated Successfully]\n")

# ==========================================
# AUTO-SAVE TO CSV (Ensures Block 2 Always Works)
# ==========================================
print(f"Saving latest data to: {FOLDER}")
df_hitters_proj.to_csv(FOLDER / 'Fangraphs_Hitter_Projections_ROS.csv', index=False)
df_pitchers_proj.to_csv(FOLDER / 'Fangraphs_Pitcher_Projections_ROS.csv', index=False)
df_rosters.to_csv(FOLDER / 'espn_current_rosters.csv', index=False)
df_current.to_csv(FOLDER / 'current_team_stats.csv', index=False)
df_fa.to_csv(FOLDER / 'espn_free_agents.csv', index=False)

print("✓ All local files synchronized. Proceeding to Analysis...")

--- STARTING REST-OF-SEASON (ROS) ANALYSIS FOR Budget Ballers ---

[Updating Data Pipelines...]
Fetching FanGraphs Projections (steamer_ros)...
 -> Pulling hitters...
 -> Pulling pitchers...
✓ FanGraphs data successfully loaded.
Connecting to ESPN Fantasy API for League 283187668...
✓ Successfully connected to: Special League Mixed League
Connecting to ESPN for Real-Time Free Agents...
✓ Found 200 real ESPN Free Agents with position data.
[Data Pipelines Updated Successfully]

Saving latest data to: C:\Users\seth.arias\Documents\Python Projects\Fantasy_Baseball_Model
✓ All local files synchronized. Proceeding to Analysis...


In [2]:
# ==========================================
# 2. DATA LOADING & CLEANING
# ==========================================
try:
    # Use the Pathlib division operator (/) to safely construct file paths
    df_rosters = pd.read_csv(FOLDER / 'espn_current_rosters.csv')
    df_hitters = pd.read_csv(FOLDER / 'Fangraphs_Hitter_Projections_ROS.csv')
    df_pitchers = pd.read_csv(FOLDER / 'Fangraphs_Pitcher_Projections_ROS.csv')
    df_current = pd.read_csv(FOLDER / 'current_team_stats.csv')
    print("✓ Success: Projections, Rosters, and Current YTD Stats loaded.")
except Exception as e:
    print(f"X Error: {e}")
    raise

# Clean hidden spaces from team names to ensure perfect merges
df_rosters['Team'] = df_rosters['Team'].astype(str).str.strip()
df_current['Team'] = df_current['Team'].astype(str).str.strip()

import unicodedata

def aggressive_clean(name):
    if not isinstance(name, str): return ""
    # Remove Accents (e.g., José -> Jose)
    name = "".join(c for c in unicodedata.normalize('NFD', name) if unicodedata.category(c) != 'Mn')
    # Lowercase, strip suffixes, and remove punctuation/dots
    name = name.lower().replace(' jr.', '').replace(' sr.', '').replace(' ii', '').replace(' iii', '').replace('.', '').strip()
    return name

for df in [df_rosters, df_hitters, df_pitchers]:
    name_col = 'Player' if 'Player' in df.columns else 'Name'
    df['Clean_Name'] = df[name_col].apply(aggressive_clean)

# Define target columns
hit_stats = ['R', 'HR', 'RBI', 'SB', 'OPS', 'PA']
pit_stats = ['IP', 'QS', 'SV', 'ER', 'H', 'BB']

# Vectorized metric cleaning (Replacing the slow .apply() loop)
for col in hit_stats:
    df_hitters[col] = pd.to_numeric(df_hitters[col].astype(str).str.replace('%', ''), errors='coerce').fillna(0.0)
for col in pit_stats:
    df_pitchers[col] = pd.to_numeric(df_pitchers[col].astype(str).str.replace('%', ''), errors='coerce').fillna(0.0)
# Since espn_pull.py already filtered the IL, df_rosters IS our active roster.
active_roster_full = df_rosters.copy()

print(f"ℹ Total Active Players Loaded: {len(active_roster_full)}")

✓ Success: Projections, Rosters, and Current YTD Stats loaded.
ℹ Total Active Players Loaded: 328


In [3]:
# ==========================================
# 3. CORE CALCULATION ENGINE (YTD + ROS)
# ==========================================
def get_league_stats(rosters_df):
    """
    Groups rosters by team, merges with ROS projections,
    adds current YTD totals, and calculates Roto points.
    """
    # 1. Map Projected Stats (Rest of Season)
    merged_h = pd.merge(
        rosters_df,
        df_hitters[['Clean_Name', 'R', 'HR', 'RBI', 'SB', 'OPS', 'PA']],
        on='Clean_Name', how='left'
    ).fillna(0)
    merged_p = pd.merge(
        rosters_df,
        df_pitchers[['Clean_Name', 'IP', 'QS', 'SV', 'ER', 'H', 'BB']],
        on='Clean_Name', how='left'
    ).fillna(0)

    # Aggregate Hitting Projections
    t_hit = merged_h.groupby('Team')[['R', 'HR', 'RBI', 'SB', 'PA']].sum().reset_index()
    merged_h['OPS_num'] = merged_h['OPS'] * merged_h['PA']
    t_hit = pd.merge(t_hit, merged_h.groupby('Team')['OPS_num'].sum().reset_index(), on='Team')

    # Aggregate Pitching Projections
    t_pit = merged_p.groupby('Team')[['IP', 'QS', 'SV', 'ER', 'H', 'BB']].sum().reset_index()

    # Combine into one DataFrame
    proj_stats = pd.merge(t_hit, t_pit, on='Team')

    # FIX: Use rename() dict instead of positional column assignment.
    # The old approach (proj_stats.columns = [...]) silently corrupts names if
    # an upstream merge ever produces unexpected _x/_y suffix columns.
    proj_stats = proj_stats.rename(
        columns={col: f"{col}_proj" for col in proj_stats.columns if col != 'Team'}
    )

    # 2. Merge Projections with Current YTD Stats
    stats = pd.merge(df_current, proj_stats, on='Team')

    # 3. Calculate End of Season (EOS) Totals
    stats['R_total']   = stats['R_curr']   + stats['R_proj']
    stats['HR_total']  = stats['HR_curr']  + stats['HR_proj']
    stats['RBI_total'] = stats['RBI_curr'] + stats['RBI_proj']
    stats['SB_total']  = stats['SB_curr']  + stats['SB_proj']

    # Weighted OPS
    stats['PA_total']  = stats['PA_curr']  + stats['PA_proj']
    stats['OPS_total'] = (
        (stats['OPS_curr'] * stats['PA_curr']) + stats['OPS_num_proj']
    ) / stats['PA_total'].replace(0, 1)

    # Pitching counting stats
    stats['IP_total'] = stats['IP_curr'] + stats['IP_proj']
    stats['QS_total'] = stats['QS_curr'] + stats['QS_proj']
    stats['SV_total'] = stats['SV_curr'] + stats['SV_proj']

    # Pitching ratios — accumulate raw components, then divide (volume-weighted)
    stats['ER_total']   = stats['ER_curr'] + stats['ER_proj']
    stats['H_total']    = stats['H_curr']  + stats['H_proj']
    stats['BB_total']   = stats['BB_curr'] + stats['BB_proj']
    stats['ERA_total']  = (stats['ER_total'] / stats['IP_total'].replace(0, 1)) * 9
    stats['WHIP_total'] = (stats['H_total'] + stats['BB_total']) / stats['IP_total'].replace(0, 1)

    # 4. Rank categories
    high_is_better = ['R_total', 'HR_total', 'RBI_total', 'SB_total',
                      'OPS_total', 'IP_total', 'QS_total', 'SV_total']
    for s in high_is_better:
        stats[f'{s}_Pts'] = stats[s].rank(method='average')

    stats['ERA_total_Pts']  = stats['ERA_total'].rank(method='average', ascending=False)
    stats['WHIP_total_Pts'] = stats['WHIP_total'].rank(method='average', ascending=False)

    stats['Total_Points'] = stats[[c for c in stats.columns if '_Pts' in c]].sum(axis=1)
    return stats


In [4]:
# ==========================================
# 5. PHASE 1: END OF SEASON STANDINGS
# ==========================================
# Calculate the baseline using the active roster
baseline_stats = get_league_stats(active_roster_full)

# Updated display columns to match the '_total' suffix from the calculation engine
display_cols = [
    'Team', 'Total_Points', 'R_total', 'HR_total', 'RBI_total', 
    'SB_total', 'OPS_total', 'IP_total', 'QS_total', 'SV_total', 
    'ERA_total', 'WHIP_total'
]

# Sort and 1-index the standings for human readability
standings = baseline_stats[display_cols].sort_values(by='Total_Points', ascending=False).reset_index(drop=True)
standings.index += 1

# Vectorized rounding for counting stats
int_cols = ['R_total', 'HR_total', 'RBI_total', 'SB_total', 'IP_total', 'QS_total', 'SV_total']
standings[int_cols] = standings[int_cols].round(0).astype(int)

print("\n--- 1. PROJECTED END OF SEASON STANDINGS (YTD + ROS) ---")

# Clean up column names for the final display table only
standings_clean = standings.rename(columns={c: c.replace('_total', '') for c in standings.columns})

display(standings_clean.style.format({
    'Total_Points': '{:.1f}', 
    'OPS': '{:.3f}', 
    'ERA': '{:.2f}', 
    'WHIP': '{:.2f}'
}))


--- 1. PROJECTED END OF SEASON STANDINGS (YTD + ROS) ---


,Team,Total_Points,R,HR,RBI,SB,OPS,IP,QS,SV,ERA,WHIP
1,Exit Velo Vets,96.0,1043,359,1017,148,0.778,1628,114,120,3.68,1.22
2,Budget Ballers,80.0,1024,326,1064,124,0.779,1742,118,78,3.84,1.27
3,Haders Gonna Hade,78.0,1023,327,1003,141,0.780,1603,103,119,3.79,1.25
4,Corbin Barrels,74.0,1019,254,925,170,0.763,1771,131,59,3.75,1.26
5,brew crew,72.0,978,232,885,196,0.773,1916,137,47,3.87,1.23
6,Rebels,68.0,935,258,942,131,0.760,1677,114,72,3.65,1.21
7,Judge Judy and Executioner,64.0,989,314,1024,141,0.778,1411,85,102,3.88,1.25
8,adam's Astounding Team,59.0,1016,291,995,188,0.762,1525,92,27,3.75,1.25
9,Spencer Steer for MVP,58.0,912,233,843,184,0.748,1656,113,80,3.76,1.23
10,The Roman Empire Strikes Back,54.0,998,296,914,155,0.776,1655,105,36,3.89,1.28


In [5]:
# ==========================================
# 6. PHASE 2: OPTIMIZED SWAPS (STRICT LEGAL)
# ==========================================

# --- 1. INITIALIZATION & DEDUPLICATION ---
active_roster_full = df_rosters.copy()
my_players = active_roster_full[active_roster_full['Team'] == MY_TEAM_NAME].copy()

# Load official ESPN Free Agent list
df_espn_fa = pd.read_csv(FOLDER / 'espn_free_agents.csv')
df_espn_fa['Clean_Name'] = df_espn_fa['Player'].apply(aggressive_clean)

# Deduplicate by name — prevents a prospect from inheriting a star's projections
df_espn_fa = df_espn_fa.drop_duplicates(subset=['Clean_Name'], keep='first')

# FIX: Rename ESPN columns before merging to eliminate _x/_y suffix ambiguity.
# Explicitly labelling these 'ESPN_Positions' and 'ESPN_Player' means the code
# never has to guess which suffix column to use — it always reads the right one.
df_espn_fa_clean = df_espn_fa.rename(columns={'Positions': 'ESPN_Positions', 'Player': 'ESPN_Player'})

top_fa_h = pd.merge(df_espn_fa_clean, df_hitters, on='Clean_Name', how='inner').head(100)
top_fa_p = pd.merge(df_espn_fa_clean, df_pitchers, on='Clean_Name', how='inner').head(100)

base_stats = baseline_stats.set_index('Team').loc[MY_TEAM_NAME]

# FIX: Build name-lookup sets ONCE before any loop.
# The old code rebuilt a NumPy array on every outer iteration via .values,
# making membership checks O(n) per call instead of O(1).
hitter_name_set  = set(df_hitters['Clean_Name'])
pitcher_name_set = set(df_pitchers['Clean_Name'])

# --- 2. THE RIGOROUS GATEKEEPER ---
def is_eligible(add_positions, drop_slot):
    """
    Enforces strict ESPN roster rules.
    Blocks out-of-position swaps (e.g. 3B-only player into a 1B slot).
    """
    slot = str(drop_slot).strip()
    if pd.isna(slot) or slot in ['BE', 'UTIL', 'DH', 'P']:
        return True

    raw_string = str(add_positions).replace('/', ' ')
    player_pos_set = set(raw_string.split())

    if slot in ['1B/3B', 'CI']:
        return '1B' in player_pos_set or '3B' in player_pos_set
    if slot in ['2B/SS', 'MI']:
        return '2B' in player_pos_set or 'SS' in player_pos_set

    return slot in player_pos_set

def get_impact_string(old_row, new_row):
    impacts = []
    pt_cols = [c for c in baseline_stats.columns if '_total_Pts' in c]
    for col in pt_cols:
        diff = new_row[col] - old_row[col]
        if diff != 0:
            label = col.replace('_total_Pts', '')
            impacts.append(f"{'+' if diff > 0 else ''}{round(diff, 1)} {label}")
    return ", ".join(impacts) if impacts else "Lateral move"

print(f"Analyzing {len(my_players)} spots for strict legal swaps...")

h_results, p_results = [], []

# --- 3. THE SIMULATION ---
for _, drop in my_players.iterrows():
    # FIX: O(1) set lookup instead of O(n) array scan rebuilt each iteration
    is_hitter = drop['Clean_Name'] in hitter_name_set
    fa_pool   = top_fa_h if is_hitter else top_fa_p

    # FIX: Pre-filter by slot eligibility BEFORE running any simulations.
    # This eliminates the gatekeeper check from the inner loop entirely and
    # reduces the simulation count to only candidates that are actually legal.
    eligible_fas = fa_pool[fa_pool['ESPN_Positions'].apply(
        lambda pos: is_eligible(pos, drop['Lineup_Slot'])
    )]

    for _, add in eligible_fas.iterrows():
        # Column names are now always unambiguous — no conditional suffix logic needed
        add_pos  = add['ESPN_Positions']
        add_name = add['ESPN_Player']

        # Simulate the swap
        df_temp = active_roster_full[
            active_roster_full['Clean_Name'] != drop['Clean_Name']
        ].copy()
        new_row = pd.DataFrame([{
            'Team':        MY_TEAM_NAME,
            'Clean_Name':  add['Clean_Name'],
            'Status':      'Rostered',
            'Lineup_Slot': drop['Lineup_Slot']
        }])
        df_temp = pd.concat([df_temp, new_row], ignore_index=True)

        sim_stats  = get_league_stats(df_temp).set_index('Team').loc[MY_TEAM_NAME]
        point_gain = sim_stats['Total_Points'] - base_stats['Total_Points']

        if point_gain > 0:
            res = {
                'Add':      add_name,
                'Drop':     drop['Player'],
                'Slot':     drop['Lineup_Slot'],
                'Net Gain': round(point_gain, 1),
                'Details':  get_impact_string(base_stats, sim_stats)
            }
            if is_hitter: h_results.append(res)
            else:         p_results.append(res)

# --- 4. DISPLAY ---
print("\n--- 2. PROFITABLE LEGAL HITTER SWAPS ---")
if h_results:
    display(pd.DataFrame(h_results).sort_values(by='Net Gain', ascending=False).head(10))
else:
    print("No legal hitter swaps found that improve your total Roto points.")

print("\n--- 3. PROFITABLE LEGAL PITCHER SWAPS ---")
if p_results:
    display(pd.DataFrame(p_results).sort_values(by='Net Gain', ascending=False).head(10))
else:
    print("No legal pitcher swaps found that improve your total Roto points.")


Analyzing 27 spots for strict legal swaps...

--- 2. PROFITABLE LEGAL HITTER SWAPS ---


,Add,Drop,Slot,Net Gain,Details
3,Max Muncy,Miguel Vargas,1B,2.0,"+1.0 R, +1.0 HR"
4,Max Muncy,Miguel Vargas,1B,2.0,"+1.0 R, +1.0 HR"
0,Andres Gimenez,Colson Montgomery,2B/SS,1.0,+1.0 SB
1,Ezequiel Tovar,Colson Montgomery,2B/SS,1.0,+1.0 OPS
2,Colt Keith,Colson Montgomery,2B/SS,1.0,+1.0 OPS
5,Nolan Schanuel,Miguel Vargas,1B,1.0,+1.0 OPS
6,Colt Keith,Miguel Vargas,1B,1.0,+1.0 OPS



--- 3. PROFITABLE LEGAL PITCHER SWAPS ---


,Add,Drop,Slot,Net Gain,Details
6,Steven Matz,Adrian Morejon,RP,2.0,"+1.0 IP, +1.0 QS"
0,Grayson Rodriguez,Seth Lugo,SP,1.0,+1.0 ERA
10,Landen Roupp,Slade Cecconi,SP,1.0,+1.0 ERA
17,Robby Snelling,Slade Cecconi,SP,1.0,+1.0 ERA
16,Braxton Garrett,Slade Cecconi,SP,1.0,+1.0 ERA
15,Sean Manaea,Slade Cecconi,SP,1.0,+1.0 ERA
14,Grayson Rodriguez,Slade Cecconi,SP,1.0,+1.0 ERA
13,Max Meyer,Slade Cecconi,SP,1.0,+1.0 ERA
12,Michael McGreevy,Slade Cecconi,SP,1.0,+1.0 ERA
11,Kyle Harrison,Slade Cecconi,SP,1.0,+1.0 ERA


In [6]:
# ==========================================
# 7. PHASE 3: MANUAL TRADE & SWAP EVALUATOR
# ==========================================

# --- 1. ENTER YOUR PROPOSED MOVES HERE ---
PLAYERS_TO_ACQUIRE       = ["Brendan Donovan"]
PLAYERS_TO_DROP_OR_TRADE = ["Jorge Polanco"]

# Set to a team name string for a trade, or None for a waiver pickup
TRADE_PARTNER = None

# -----------------------------------------

print(f"\n--- EVALUATING CUSTOM SWAP ---")
print(f"Acquiring: {', '.join(PLAYERS_TO_ACQUIRE)}")
print(f"Shipping Out: {', '.join(PLAYERS_TO_DROP_OR_TRADE)}")
if TRADE_PARTNER: print(f"Trade Partner: {TRADE_PARTNER}")

# FIX: Clean input names before any lookup.
# Player names typed here are raw (e.g. "José Ramírez"), but the roster
# DataFrame uses Clean_Name (accent-stripped, lowercased, suffix-removed).
# Without cleaning, the lookup silently fails and the simulation runs on the
# unmodified roster — returning a 0.0 point change with no warning.
acquire_cleaned = [aggressive_clean(p) for p in PLAYERS_TO_ACQUIRE]
drop_cleaned    = [aggressive_clean(p) for p in PLAYERS_TO_DROP_OR_TRADE]

df_sim = active_roster_full.copy()

# 2. Process players being given up
for raw, cleaned in zip(PLAYERS_TO_DROP_OR_TRADE, drop_cleaned):
    mask = df_sim['Clean_Name'] == cleaned
    if mask.any():
        if TRADE_PARTNER:
            df_sim.loc[mask, 'Team'] = TRADE_PARTNER
        else:
            df_sim = df_sim[~mask]
    else:
        print(f"⚠️  Warning: '{raw}' not found on your active roster (checked as '{cleaned}').")

# 3. Process players being acquired
for raw, cleaned in zip(PLAYERS_TO_ACQUIRE, acquire_cleaned):
    mask = df_sim['Clean_Name'] == cleaned
    if mask.any():
        df_sim.loc[mask, 'Team'] = MY_TEAM_NAME
    else:
        # Free agent not yet on any roster — append with cleaned name so
        # get_league_stats() can match them against the projections
        new_row = pd.DataFrame([{
            'Team':        MY_TEAM_NAME,
            'Clean_Name':  cleaned,
            'Status':      'Rostered',
            'Lineup_Slot': 'BE'
        }])
        df_sim = pd.concat([df_sim, new_row], ignore_index=True)

# 4. Run the simulation
sim_stats = get_league_stats(df_sim)

# 5. Extract Before & After
base_my_team = baseline_stats.set_index('Team').loc[MY_TEAM_NAME]
sim_my_team  = sim_stats.set_index('Team').loc[MY_TEAM_NAME]

gain       = sim_my_team['Total_Points'] - base_my_team['Total_Points']
impact_str = get_impact_string(base_my_team, sim_my_team)

print(f"\n[ RESULTS FOR {MY_TEAM_NAME} ]")
print(f"Net Point Change: {'+' if gain > 0 else ''}{gain:.1f} Points")
print(f"Category Shifts:  {impact_str}")

# 6. New standings preview (top 5)
print("\n--- NEW STANDINGS PREVIEW (TOP 5) ---")
display_cols = [
    'Team', 'Total_Points', 'R_total', 'HR_total', 'RBI_total',
    'SB_total', 'OPS_total', 'IP_total', 'QS_total', 'SV_total',
    'ERA_total', 'WHIP_total'
]
sim_standings = (
    sim_stats[display_cols]
    .sort_values(by='Total_Points', ascending=False)
    .reset_index(drop=True)
)
sim_standings.index += 1

int_cols = ['R_total', 'HR_total', 'RBI_total', 'SB_total', 'IP_total', 'QS_total', 'SV_total']
sim_standings[int_cols] = sim_standings[int_cols].round(0).astype(int)

sim_standings_clean = sim_standings.rename(columns={c: c.replace('_total', '') for c in sim_standings.columns})
display(sim_standings_clean.head(5).style.format({
    'Total_Points': '{:.1f}', 'OPS': '{:.3f}', 'ERA': '{:.2f}', 'WHIP': '{:.2f}'
}))



--- EVALUATING CUSTOM SWAP ---
Acquiring: Brendan Donovan
Shipping Out: Jorge Polanco

[ RESULTS FOR Budget Ballers ]
Net Point Change: 0.0 Points
Category Shifts:  Lateral move

--- NEW STANDINGS PREVIEW (TOP 5) ---


,Team,Total_Points,R,HR,RBI,SB,OPS,IP,QS,SV,ERA,WHIP
1,Exit Velo Vets,96.0,1043,359,1017,148,0.778,1628,114,120,3.68,1.22
2,Budget Ballers,80.0,1035,320,1054,127,0.779,1742,118,78,3.84,1.27
3,Haders Gonna Hade,78.0,1023,327,1003,141,0.780,1603,103,119,3.79,1.25
4,Corbin Barrels,74.0,1019,254,925,170,0.763,1771,131,59,3.75,1.26
5,brew crew,72.0,978,232,885,196,0.773,1916,137,47,3.87,1.23


In [7]:
# ==========================================
# 8. PHASE 4: EMPTY SLOT FILLER (BEST FA BY POSITION)
# ==========================================
TARGET_POSITION      = "RP"  # e.g. 'OF', 'SP', '1B', 'C'
MAX_CANDIDATES_TO_TEST = 50
TOP_N_RESULTS          = 10

print(f"\n--- FINDING BEST AVAILABLE '{TARGET_POSITION}' FOR {MY_TEAM_NAME} ---")

# 1. Pitcher or hitter?
pitching_positions = ['SP', 'RP', 'P']
is_pitcher = TARGET_POSITION in pitching_positions

# FIX: Source free agents from df_espn_fa, not df_rosters.
# df_rosters only contains actively rostered players (Status = 'Rostered').
# No player in that DataFrame will ever have Status = 'Available', so the old
# filter silently returned an empty list on every single run.
# df_espn_fa is the correct source — it was built from the ESPN waiver wire.
available_fas = df_espn_fa[
    df_espn_fa['Positions'].astype(str).str.contains(
        rf'\b{TARGET_POSITION}\b', regex=True, na=False
    )
].copy()

fa_names = available_fas['Clean_Name'].tolist()

if not fa_names:
    print(f"⚠️  No free agents with '{TARGET_POSITION}' eligibility found in espn_free_agents.csv.")
else:
    # 3. Sort by projected volume to avoid simulating fringe players
    if is_pitcher:
        candidates = (
            df_pitchers[df_pitchers['Clean_Name'].isin(fa_names)]
            .sort_values(by='IP', ascending=False)
            .head(MAX_CANDIDATES_TO_TEST)
        )
    else:
        candidates = (
            df_hitters[df_hitters['Clean_Name'].isin(fa_names)]
            .sort_values(by='PA', ascending=False)
            .head(MAX_CANDIDATES_TO_TEST)
        )

    print(f"Simulating adding the top {len(candidates)} projected {TARGET_POSITION}s to your roster...")

    fill_results    = []
    base_my_team    = baseline_stats.set_index('Team').loc[MY_TEAM_NAME]

    for _, add_player in candidates.iterrows():
        df_temp = active_roster_full.copy()
        new_row = pd.DataFrame([{
            'Team':        MY_TEAM_NAME,
            'Clean_Name':  add_player['Clean_Name'],
            'Status':      'Rostered',
            'Lineup_Slot': TARGET_POSITION
        }])
        df_temp = pd.concat([df_temp, new_row], ignore_index=True)

        new_stats = get_league_stats(df_temp).set_index('Team').loc[MY_TEAM_NAME]
        gain      = new_stats['Total_Points'] - base_my_team['Total_Points']

        fill_results.append({
            'Add':      add_player.get('Player', add_player['Clean_Name']),
            'Net Gain': round(gain, 1),
            'Details':  get_impact_string(base_my_team, new_stats)
        })

    if fill_results:
        df_results = (
            pd.DataFrame(fill_results)
            .sort_values(by='Net Gain', ascending=False)
            .head(TOP_N_RESULTS)
        )
        display(df_results)
    else:
        print(f"⚠️  No {TARGET_POSITION}s with matching projections found.")



--- FINDING BEST AVAILABLE 'RP' FOR Budget Ballers ---
Simulating adding the top 30 projected RPs to your roster...


,Add,Net Gain,Details
0,Steven Matz,3.0,"+2.0 IP, +1.0 QS"
10,Cole Sands,2.0,"+1.0 IP, +1.0 SV"
27,Garrett Cleavinger,2.0,"+1.0 IP, +1.0 SV"
25,Chris Martin,2.0,"+1.0 IP, +1.0 SV"
21,Taylor Rogers,2.0,"+1.0 IP, +1.0 SV"
20,Andrew Kittredge,2.0,"+1.0 IP, +1.0 SV"
16,Graham Ashcraft,2.0,"+1.0 IP, +1.0 SV"
12,Luke Weaver,2.0,"+1.0 IP, +1.0 SV"
11,Shawn Armstrong,2.0,"+1.0 IP, +1.0 SV"
15,Phil Maton,2.0,"+1.0 IP, +1.0 SV"
